# GOT-OCR Test Notebook

Objectives:
1. Load the `stepfun-ai/GOT-OCR-2.0-hf` model.
2. Replicate the structure of `docling.ipynb` and `doctr.ipynb`.
3. Test on all imported data (CIN and Papyrus documents).

In [ ]:
import io
import time
from pathlib import Path

import fitz  # PyMuPDF
import matplotlib.pyplot as plt
import torch
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor

In [ ]:
MODEL_ID = "stepfun-ai/GOT-OCR-2.0-hf"
device = "cpu"  # Defaulting to CPU based on previous tests

print(f"Loading model: {MODEL_ID}")
try:
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    model = AutoModelForImageTextToText.from_pretrained(MODEL_ID).to(device)
    print("✅ Model loaded successfully")
except Exception as e:
    print(f"❌ Error loading model: {e}")

In [ ]:
def run_got_ocr(image_source):
    """Run GOT-OCR on a file path or PIL Image."""
    start_time = time.time()
    try:
        if isinstance(image_source, str | Path):
            # print(f"Processing {image_source}...") # moved to main loop for cleaner output
            image = Image.open(image_source).convert("RGB")
        else:
            # Assume it's already a PIL image
            image = image_source

        inputs = processor(image, return_tensors="pt").to(device)

        with torch.no_grad():
            generate_ids = model.generate(
                **inputs,
                do_sample=False,
                tokenizer=processor.tokenizer,
                stop_strings="<|im_end|>",
                max_new_tokens=4096,
            )

        result = processor.decode(
            generate_ids[0, inputs["input_ids"].shape[1] :], skip_special_tokens=True
        )
        inference_time = time.time() - start_time
        return result, inference_time
    except Exception as e:
        print(f"❌ Error: {e}")
        return None, 0


def process_and_display(file_path):
    """Handle both images and PDFs."""
    file_path = Path(file_path)
    if not file_path.exists():
        print(f"❌ File not found: {file_path}")
        return

    print(f"\n{'='*20} Processing: {file_path.name} {'='*20}")

    if file_path.suffix.lower() == ".pdf":
        try:
            doc = fitz.open(file_path)
            for page_num, page in enumerate(doc):
                print(f"\n--- Page {page_num + 1} ---")
                pix = page.get_pixmap()
                img_data = pix.tobytes("png")
                image = Image.open(io.BytesIO(img_data)).convert("RGB")

                # Display image
                plt.figure(figsize=(10, 10))
                plt.imshow(image)
                plt.title(f"{file_path.name} - Page {page_num + 1}")
                plt.axis("off")
                plt.show()

                # Run OCR
                text, duration = run_got_ocr(image)
                if text:
                    print(f"✅ OCR Completed in {duration:.2f}s")
                    print("OCR Result:")
                    print("-" * 20)
                    print(text[:1000] + "..." if len(text) > 1000 else text)
                    print("-" * 20)
        except Exception as e:
            print(f"❌ Error processing PDF {file_path}: {e}")

    else:  # Image file
        try:
            # Display image
            image = Image.open(file_path).convert("RGB")
            plt.figure(figsize=(10, 10))
            plt.imshow(image)
            plt.title(file_path.name)
            plt.axis("off")
            plt.show()

            # Run OCR
            text, duration = run_got_ocr(file_path)
            if text:
                print(f"✅ OCR Completed in {duration:.2f}s")
                print("OCR Result:")
                print("-" * 20)
                print(text[:1000] + "..." if len(text) > 1000 else text)
                print("-" * 20)
        except Exception as e:
            print(f"❌ Error processing image {file_path}: {e}")

In [ ]:
data_dir = Path("../data")

# Define all files to process
files_to_process = [
    # Standard test image
    "image_ocr.jpg",
    # CIN Files
    "trash_cin.jpg",
    "trash_cin_1.jpg",
    "trash_cin_1.pdf",
    "trash_cin_2.jpg",
    "trash_cin_2.pdf",
    # Papyrus Files
    "trash_papyrus_1.pdf",
    "trash_papyrus_2.pdf",
]

print(f"Found {len(files_to_process)} files to process.")

for filename in files_to_process:
    process_and_display(data_dir / filename)